# 06 — Geographic Model Training

## 1. Introduction

### Objective

The objective of this notebook is to evaluate whether geographic information improves the prediction of Airbnb nightly prices in Milan compared with the structural baseline model developed in Notebook 03.

The baseline model uses seven structural features and achieved the following results on the held-out test set:

- **Mean Absolute Error (MAE):** 70.70
- **R² score:** 0.227

### Experimental Strategy

We will compare four geographic feature configurations, each combined with the original seven structural predictors:

1. **Distance:** Distance from Milan Cathedral (Duomo).
2. **Coordinates:** Latitude and longitude.
3. **Neighbourhood:** Administrative neighbourhood.
4. **All geographic features:** Distance, coordinates, and neighbourhood.

To ensure a fair comparison, we will use the same regression algorithm, hyperparameters, train/test split strategy, and cross-validation settings across experiments.

Model selection will rely exclusively on cross-validation results from the training set. The held-out test set will be used only for the final evaluation of the selected model.

All preprocessing steps will be included within the model pipeline to prevent data leakage during cross-validation.

### Notebook Outline

1. Introduction
2. Data Loading and Preparation
3. Train/Test Split
4. Preprocessing and Model Pipeline
5. Geographic Feature Experiments
6. Cross-Validation Results
7. Final Model Evaluation
8. Comparison with Model 1
9. Final Summary

## 2. Data Loading and Preparation

In [1]:
import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
# Load the original Inside Airbnb dataset
df = pd.read_csv("../data/raw/listings.csv.gz")

# Check dataset dimensions
print("Original dataset shape:", df.shape)

# Preview the first five rows
display(df.head())

Original dataset shape: (25338, 90)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,23986,https://www.airbnb.com/rooms/23986,20260919134113,2026-09-21,NaN,""" Characteristic Milanese flat""",I look forward to welcoming you in my flat; it...,NaN,https://a0.muscache.com/pictures/623d63f8-56cf...,95941,...,4.73,4.69,4.46,NaN,NaN,1,1,0,0,0.15
1,40470,https://www.airbnb.com/rooms/40470,20260919134113,2026-09-20,NaN,Giacinto Cosy & clean flat near MM1,<b>The space</b><br />If you want to visit Mil...,NaN,https://a0.muscache.com/pictures/891684/01c17b...,174203,...,4.88,4.45,4.52,IT015146C2YDCLSU8G,NaN,2,2,0,0,0.23
2,46536,https://www.airbnb.com/rooms/46536,20260919134113,2026-09-20,NaN,Nico & Cinzia's Pink Suite!,Over the international fair in April we rent f...,NaN,https://a0.muscache.com/pictures/4eb8e0f5-e17b...,138683,...,4.92,4.36,4.59,IT015209C2BBKGCMT7,NaN,1,1,0,0,0.22
3,84741,https://www.airbnb.com/rooms/84741,20260919134113,2026-09-21,NaN,Nice room with private bathroom,Independent room with private bathroom.<br />C...,NaN,https://a0.muscache.com/pictures/0b34089b-4d58...,443989,...,4.89,4.18,4.64,"I applied for a CIN, but haven’t received it yet",NaN,3,1,2,0,3.75
4,88130,https://www.airbnb.com/rooms/88130,20260919134113,2026-09-21,NaN,Great comfortable & quite APT. x 4 people!,Our bright and comfortable apartment for 3+ 1 ...,NaN,https://a0.muscache.com/pictures/606790/408e10...,481475,...,5.00,4.43,4.71,NaN,NaN,1,1,0,0,0.04


In [3]:
# Convert nightly prices to numeric values
df["price_numeric"] = pd.to_numeric(
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

# Apply the same filtering criteria as the baseline model
df = df.loc[
    df["room_type"].isin(["Entire home/apt", "Private room"])
    & df["price_numeric"].gt(0)
    & df["price_numeric"].le(1000)
].copy()

# Check the filtered dataset
print("Filtered dataset shape:", df.shape)
print("\nRoom type distribution:")
print(df["room_type"].value_counts())

Filtered dataset shape: (21314, 91)

Room type distribution:
room_type
Entire home/apt    18877
Private room        2437
Name: count, dtype: int64


In [4]:
def add_bathroom_features(data):
    """
    Extract bathroom count and privacy type
    from the bathrooms_text column.
    """
    result = data.copy()

    text = result["bathrooms_text"].astype("string")

    numbers = pd.to_numeric(
        text.str.extract(
            r"(\d+(?:\.\d+)?)",
            expand=False
        ),
        errors="coerce"
    )

    half_bath = (
        text.str.contains("half-bath", case=False, na=False)
        & numbers.isna()
    )

    result["bathrooms_count"] = numbers.mask(half_bath, 0.5)

    text_lower = text.str.lower()

    result["bathroom_type"] = "unspecified"

    result.loc[
        text_lower.str.contains("private", na=False),
        "bathroom_type"
    ] = "private"

    result.loc[
        text_lower.str.contains("shared", na=False),
        "bathroom_type"
    ] = "shared"

    result.loc[
        text_lower.isna(),
        "bathroom_type"
    ] = "missing"

    return result


df = add_bathroom_features(df)

print("Missing bathroom counts:", df["bathrooms_count"].isna().sum())
print("\nBathroom types:")
print(df["bathroom_type"].value_counts())

Missing bathroom counts: 3

Bathroom types:
bathroom_type
unspecified    19137
private         1266
shared           908
missing            3
Name: count, dtype: int64


In [5]:
# Structural features used in Model 1
structural_features = [
    "room_type",
    "property_type",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms_count",
    "bathroom_type"
]

# Geographic features evaluated in Model 2
geographic_features = [
    "latitude",
    "longitude",
    "neighbourhood_cleansed",
    "distance_duomo_km"
]

print("Structural features:", len(structural_features))
print("Geographic features:", len(geographic_features))
print("Total features:", len(structural_features + geographic_features))

Structural features: 7
Geographic features: 4
Total features: 11


In [6]:
# Duomo di Milano coordinates
DUOMO_LAT = 45.4642
DUOMO_LON = 9.1916
EARTH_RADIUS_KM = 6371.0

# Convert coordinates from degrees to radians
lat1 = np.radians(df["latitude"])
lon1 = np.radians(df["longitude"])

lat2 = np.radians(DUOMO_LAT)
lon2 = np.radians(DUOMO_LON)

# Apply the Haversine formula
delta_lat = lat1 - lat2
delta_lon = lon1 - lon2

a = (
    np.sin(delta_lat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2)
    * np.sin(delta_lon / 2) ** 2
)

df["distance_duomo_km"] = (
    2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))
)

# Validate the engineered feature
print("Missing distances:", df["distance_duomo_km"].isna().sum())
print("\nDistance from Duomo (km):")
print(df["distance_duomo_km"].describe().round(2))

Missing distances: 0

Distance from Duomo (km):
count    21314.00
mean         2.94
std          1.56
min          0.03
25%          1.80
50%          2.76
75%          3.78
max          9.84
Name: distance_duomo_km, dtype: float64


In [7]:
# Check missing values in the selected features and target
selected_columns = (
    structural_features
    + geographic_features
    + ["price_numeric"]
)

missing_values = (
    df[selected_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values by column:")
print(missing_values)

print("\nTotal missing values:", missing_values.sum())

Missing values by column:
beds                      432
bedrooms                    8
bathrooms_count             3
room_type                   0
property_type               0
accommodates                0
bathroom_type               0
latitude                    0
longitude                   0
neighbourhood_cleansed      0
distance_duomo_km           0
price_numeric               0
dtype: int64

Total missing values: 443


## 3. Train/Test Split

In [8]:
# Define predictors and target
X = df[structural_features + geographic_features].copy()
y = df["price_numeric"].copy()

# Reproduce the baseline train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=df["room_type"]
)

# Check split dimensions
print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

# Check that training and test indices do not overlap
assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(df)

print("\nTrain/test split validation passed.")

Training features: (17051, 11)
Test features: (4263, 11)
Training target: (17051,)
Test target: (4263,)

Train/test split validation passed.


## 4. Preprocessing and Model Pipeline

In [9]:
class BedsImputer(BaseEstimator, TransformerMixin):
    """
    Impute missing bed counts using room-type medians
    and create a missing-value indicator.
    """

    def fit(self, X, y=None):
        self.medians_ = X.groupby("room_type")["beds"].median()
        self.global_median_ = X["beds"].median()
        return self

    def transform(self, X):
        X = X.copy()

        X["beds_missing"] = X["beds"].isna().astype(int)

        room_type_medians = X["room_type"].map(self.medians_)

        X["beds"] = (
            X["beds"]
            .fillna(room_type_medians)
            .fillna(self.global_median_)
        )

        return X

In [10]:
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """
    Group infrequent categories into 'Other',
    using frequencies learned from training data.
    """

    def __init__(self, column="property_type", min_frequency=30):
        self.column = column
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        category_counts = X[self.column].value_counts()

        self.frequent_categories_ = category_counts[
            category_counts >= self.min_frequency
        ].index

        return self

    def transform(self, X):
        X = X.copy()

        X[self.column] = X[self.column].where(
            X[self.column].isin(self.frequent_categories_),
            "Other"
        )

        return X

In [11]:
def build_preprocessing_pipeline(selected_features):
    """
    Build a preprocessing pipeline for a selected feature set.
    """

    numeric_candidates = [
        "accommodates",
        "bedrooms",
        "beds",
        "bathrooms_count",
        "beds_missing",
        "latitude",
        "longitude",
        "distance_duomo_km"
    ]

    categorical_candidates = [
        "room_type",
        "property_type",
        "bathroom_type",
        "neighbourhood_cleansed"
    ]

    # beds_missing is generated by BedsImputer
    numeric_features = [
        feature
        for feature in numeric_candidates
        if feature in selected_features or feature == "beds_missing"
    ]

    categorical_features = [
        feature
        for feature in categorical_candidates
        if feature in selected_features
    ]

    column_transformer = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ],
    sparse_threshold=0
    )

    preprocessing_pipeline = Pipeline(
        steps=[
            ("beds_imputer", BedsImputer()),
            (
                "rare_categories",
                RareCategoryGrouper(
                    column="property_type",
                    min_frequency=30
                )
            ),
            ("column_transformer", column_transformer)
        ]
    )

    return preprocessing_pipeline

In [12]:
def build_model_pipeline(selected_features):
    """
    Build a complete pipeline combining preprocessing
    and the baseline regression algorithm.
    """

    preprocessing = build_preprocessing_pipeline(selected_features)

    regressor = HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        random_state=42
    )

    model_pipeline = Pipeline(
        steps=[
            ("preprocessing", preprocessing),
            ("regressor", regressor)
        ]
    )

    return model_pipeline

In [13]:
# Build the preprocessing pipeline with all selected features
all_features = structural_features + geographic_features

preprocessing_check = build_preprocessing_pipeline(all_features)

# Fit and transform training data only
X_train_check = preprocessing_check.fit_transform(X_train, y_train)

# Inspect the transformed output
print("Transformed shape:", X_train_check.shape)
print("Sparse matrix:", sparse.issparse(X_train_check))
print("Output type:", type(X_train_check).__name__)

Transformed shape: (17051, 116)
Sparse matrix: False
Output type: ndarray


## 5. Geographic Feature Experiments

In [14]:
# Define feature configurations for model comparison
feature_experiments = {
    "Structural Baseline": structural_features,

    "Distance": structural_features + [
        "distance_duomo_km"
    ],

    "Coordinates": structural_features + [
        "latitude",
        "longitude"
    ],

    "Neighbourhood": structural_features + [
        "neighbourhood_cleansed"
    ],

    "All Geographic": structural_features + geographic_features
}

# Check the number of input features per experiment
for experiment_name, features in feature_experiments.items():
    print(f"{experiment_name}: {len(features)} features")

Structural Baseline: 7 features
Distance: 8 features
Coordinates: 9 features
Neighbourhood: 8 features
All Geographic: 11 features


## 6. Cross-Validation Results

In [15]:
# Define the cross-validation strategy
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Define evaluation metrics
scoring = {
    "MAE": "neg_mean_absolute_error",
    "R2": "r2"
}

print("Cross-validation folds:", cv.get_n_splits())
print("Evaluation metrics:", list(scoring.keys()))

Cross-validation folds: 5
Evaluation metrics: ['MAE', 'R2']


In [16]:
# Build the structural baseline model
baseline_features = feature_experiments["Structural Baseline"]

baseline_pipeline = build_model_pipeline(baseline_features)

# Evaluate the complete pipeline using training data only
baseline_cv = cross_validate(
    baseline_pipeline,
    X_train[baseline_features],
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    error_score="raise"
)

# Convert negative MAE scores to positive values
baseline_mae = -baseline_cv["test_MAE"]
baseline_r2 = baseline_cv["test_R2"]

# Display cross-validation results
print("Structural Baseline — Cross-Validation Results")
print(f"MAE: {baseline_mae.mean():.2f} ± {baseline_mae.std():.2f}")
print(f"R²:  {baseline_r2.mean():.3f} ± {baseline_r2.std():.3f}")

Structural Baseline — Cross-Validation Results
MAE: 71.48 ± 2.16
R²:  0.240 ± 0.023


In [17]:
# Experiment A: structural features + distance from Duomo
distance_features = feature_experiments["Distance"]

distance_pipeline = build_model_pipeline(distance_features)

distance_cv = cross_validate(
    distance_pipeline,
    X_train[distance_features],
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    error_score="raise"
)

# Convert negative MAE scores to positive values
distance_mae = -distance_cv["test_MAE"]
distance_r2 = distance_cv["test_R2"]

# Compare with the structural baseline
print("Distance — Cross-Validation Results")
print(f"MAE: {distance_mae.mean():.2f} ± {distance_mae.std():.2f}")
print(f"R²:  {distance_r2.mean():.3f} ± {distance_r2.std():.3f}")

print("\nChange compared with Structural Baseline:")
print(f"MAE reduction: {baseline_mae.mean() - distance_mae.mean():.2f}")
print(f"R² increase:  {distance_r2.mean() - baseline_r2.mean():+.3f}")

Distance — Cross-Validation Results
MAE: 62.03 ± 1.81
R²:  0.377 ± 0.037

Change compared with Structural Baseline:
MAE reduction: 9.45
R² increase:  +0.137


In [18]:
# Experiment B: structural features + geographic coordinates
coordinates_features = feature_experiments["Coordinates"]

coordinates_pipeline = build_model_pipeline(coordinates_features)

coordinates_cv = cross_validate(
    coordinates_pipeline,
    X_train[coordinates_features],
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    error_score="raise"
)

# Convert negative MAE scores to positive values
coordinates_mae = -coordinates_cv["test_MAE"]
coordinates_r2 = coordinates_cv["test_R2"]

# Display cross-validation results
print("Coordinates — Cross-Validation Results")
print(f"MAE: {coordinates_mae.mean():.2f} ± {coordinates_mae.std():.2f}")
print(f"R²:  {coordinates_r2.mean():.3f} ± {coordinates_r2.std():.3f}")

print("\nChange compared with Structural Baseline:")
print(f"MAE reduction: {baseline_mae.mean() - coordinates_mae.mean():.2f}")
print(f"R² increase:  {coordinates_r2.mean() - baseline_r2.mean():+.3f}")

print("\nChange compared with Distance:")
print(f"MAE reduction: {distance_mae.mean() - coordinates_mae.mean():.2f}")
print(f"R² increase:  {coordinates_r2.mean() - distance_r2.mean():+.3f}")

Coordinates — Cross-Validation Results
MAE: 60.79 ± 1.96
R²:  0.395 ± 0.037

Change compared with Structural Baseline:
MAE reduction: 10.69
R² increase:  +0.155

Change compared with Distance:
MAE reduction: 1.24
R² increase:  +0.018


In [19]:
# Experiment C: structural features + neighbourhood
neighbourhood_features = feature_experiments["Neighbourhood"]

neighbourhood_pipeline = build_model_pipeline(neighbourhood_features)

neighbourhood_cv = cross_validate(
    neighbourhood_pipeline,
    X_train[neighbourhood_features],
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    error_score="raise"
)

# Convert negative MAE scores to positive values
neighbourhood_mae = -neighbourhood_cv["test_MAE"]
neighbourhood_r2 = neighbourhood_cv["test_R2"]

# Display cross-validation results
print("Neighbourhood — Cross-Validation Results")
print(f"MAE: {neighbourhood_mae.mean():.2f} ± {neighbourhood_mae.std():.2f}")
print(f"R²:  {neighbourhood_r2.mean():.3f} ± {neighbourhood_r2.std():.3f}")

print("\nChange compared with Structural Baseline:")
print(f"MAE reduction: {baseline_mae.mean() - neighbourhood_mae.mean():.2f}")
print(f"R² increase:  {neighbourhood_r2.mean() - baseline_r2.mean():+.3f}")

print("\nChange compared with Coordinates:")
print(f"MAE reduction: {coordinates_mae.mean() - neighbourhood_mae.mean():.2f}")
print(f"R² increase:  {neighbourhood_r2.mean() - coordinates_r2.mean():+.3f}")

Neighbourhood — Cross-Validation Results
MAE: 62.80 ± 2.08
R²:  0.365 ± 0.031

Change compared with Structural Baseline:
MAE reduction: 8.68
R² increase:  +0.125

Change compared with Coordinates:
MAE reduction: -2.01
R² increase:  -0.030


In [20]:
# Experiment D: structural features + all geographic features
all_geographic_features = feature_experiments["All Geographic"]

all_geographic_pipeline = build_model_pipeline(all_geographic_features)

all_geographic_cv = cross_validate(
    all_geographic_pipeline,
    X_train[all_geographic_features],
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    error_score="raise"
)

# Convert negative MAE scores to positive values
all_geographic_mae = -all_geographic_cv["test_MAE"]
all_geographic_r2 = all_geographic_cv["test_R2"]

# Display cross-validation results
print("All Geographic — Cross-Validation Results")
print(f"MAE: {all_geographic_mae.mean():.2f} ± {all_geographic_mae.std():.2f}")
print(f"R²:  {all_geographic_r2.mean():.3f} ± {all_geographic_r2.std():.3f}")

print("\nChange compared with Structural Baseline:")
print(f"MAE reduction: {baseline_mae.mean() - all_geographic_mae.mean():.2f}")
print(f"R² increase:  {all_geographic_r2.mean() - baseline_r2.mean():+.3f}")

print("\nChange compared with Coordinates:")
print(f"MAE reduction: {coordinates_mae.mean() - all_geographic_mae.mean():.2f}")
print(f"R² increase:  {all_geographic_r2.mean() - coordinates_r2.mean():+.3f}")

All Geographic — Cross-Validation Results
MAE: 60.71 ± 1.73
R²:  0.395 ± 0.037

Change compared with Structural Baseline:
MAE reduction: 10.77
R² increase:  +0.155

Change compared with Coordinates:
MAE reduction: 0.08
R² increase:  -0.000


In [21]:
# Compare Coordinates and All Geographic on identical CV folds
fold_comparison = pd.DataFrame({
    "Fold": np.arange(1, 6),
    "Coordinates_MAE": coordinates_mae,
    "All_Geographic_MAE": all_geographic_mae
})

# Positive values indicate an improvement from adding all geographic features
fold_comparison["MAE_Reduction"] = (
    fold_comparison["Coordinates_MAE"]
    - fold_comparison["All_Geographic_MAE"]
)

print("Paired Fold Comparison:")
display(fold_comparison.round(3))

print(
    "\nFolds improved:",
    (fold_comparison["MAE_Reduction"] > 0).sum(),
    "out of 5"
)

print(
    "Mean MAE reduction:",
    round(fold_comparison["MAE_Reduction"].mean(), 3)
)

Paired Fold Comparison:


,Fold,Coordinates_MAE,All_Geographic_MAE,MAE_Reduction
0,1,62.963,62.724,0.239
1,2,58.795,58.834,-0.039
2,3,58.274,58.690,-0.415
3,4,61.107,60.776,0.330
4,5,62.830,62.533,0.297



Folds improved: 3 out of 5
Mean MAE reduction: 0.082


## 7. Final Model Evaluation

In [22]:
# Select the final model based on cross-validation results
selected_model_name = "Coordinates"
selected_features = feature_experiments[selected_model_name]

# Build a fresh, unfitted pipeline
final_model = build_model_pipeline(selected_features)

# Fit the complete pipeline on the full training set
final_model.fit(
    X_train[selected_features],
    y_train
)

print("Selected model:", selected_model_name)
print("Input features:", len(selected_features))
print("Training samples:", len(y_train))
print("Final model training completed successfully.")

Selected model: Coordinates
Input features: 9
Training samples: 17051
Final model training completed successfully.


In [23]:
# Generate predictions on the held-out test set
y_test_pred = final_model.predict(
    X_test[selected_features]
)

# Calculate final test metrics
final_test_mae = mean_absolute_error(y_test, y_test_pred)
final_test_r2 = r2_score(y_test, y_test_pred)

# Display final evaluation results
print("Final Model — Test Set Evaluation")
print("Selected model:", selected_model_name)
print("Test samples:", len(y_test))
print(f"Test MAE: {final_test_mae:.2f}")
print(f"Test R²:  {final_test_r2:.3f}")

Final Model — Test Set Evaluation
Selected model: Coordinates
Test samples: 4263
Test MAE: 61.09
Test R²:  0.368


## 8. Comparison with Model 1

In [24]:
# Compare the final test results of Model 1 and Model 2
baseline_test_mae = 70.70
baseline_test_r2 = 0.227

model_comparison = pd.DataFrame({
    "Model": [
        "Model 1 — Structural",
        "Model 2 — Coordinates"
    ],
    "Test_MAE": [
        baseline_test_mae,
        final_test_mae
    ],
    "Test_R2": [
        baseline_test_r2,
        final_test_r2
    ]
})

# Calculate improvement relative to Model 1
mae_reduction = baseline_test_mae - final_test_mae
mae_reduction_pct = (
    mae_reduction / baseline_test_mae * 100
)
r2_increase = final_test_r2 - baseline_test_r2

display(model_comparison.round(3))

print(f"Absolute MAE reduction: {mae_reduction:.2f}")
print(f"Relative MAE reduction: {mae_reduction_pct:.1f}%")
print(f"R² increase: {r2_increase:+.3f}")

,Model,Test_MAE,Test_R2
0,Model 1 — Structural,70.700,0.227
1,Model 2 — Coordinates,61.094,0.368


Absolute MAE reduction: 9.61
Relative MAE reduction: 13.6%
R² increase: +0.141


## 9. Final Summary

### Cross-Validation Findings

Five feature configurations were evaluated using 5-fold cross-validation on the training set, with the same regression algorithm and hyperparameters.

| Feature Configuration | Mean MAE | Mean R² |
|---|---:|---:|
| Structural Baseline | 71.48 | 0.240 |
| Distance | 62.03 | 0.377 |
| Coordinates | 60.79 | 0.395 |
| Neighbourhood | 62.80 | 0.365 |
| All Geographic | 60.71 | 0.395 |

All four geographic configurations outperformed the structural baseline.

The Coordinates and All Geographic configurations achieved nearly identical predictive performance. Although All Geographic produced a slightly lower mean MAE (60.71 versus 60.79), its advantage was only 0.082 across the five folds and was observed in three of the five folds.

### Model Selection

The **Coordinates** configuration was selected as the final model because it offered a simpler representation with practically equivalent cross-validation performance.

It uses seven structural features and two geographic features (`latitude` and `longitude`), without requiring distance calculations or neighbourhood encoding.

Model selection was completed using training-set cross-validation results, before evaluating the selected model on the held-out test set.

### Final Test Results

| Metric | Model 1 — Structural | Model 2 — Coordinates |
|---|---:|---:|
| Test MAE | 70.70 | 61.09 |
| Test R² | 0.227 | 0.368 |

Compared with Model 1, Model 2 achieved:

- An absolute MAE reduction of **9.61**.
- A relative MAE reduction of **13.6%**.
- An R² increase of **0.141**.

These findings indicate that geographic coordinates provide substantial additional predictive information beyond structural listing characteristics.

### Limitations

- The comparison with Model 1 assumes equivalent train/test listing membership, based on identical filtering and split settings. Listing-level membership has not been independently verified.
- Geographic coordinates may not capture all local factors influencing listing prices.
- The analysis evaluates predictive associations, not causal effects.
- The model was evaluated using a random train/test split of the available listings; performance on future listings or different data collection periods remains unknown.

### Next Steps

The next stage of the project will investigate whether listing amenities provide additional predictive value beyond structural and geographic features.

The selected Coordinates model will serve as the geographic reference for evaluating the next feature group.